# test for vorarlberg

In [5]:
import os

import pandas as pd
import numpy as np

In [6]:
data_path = "../data/"

regions = {"vor": "20241214-0617_gtfs_vor_2024", #vienna, lower austria, burgenland
           "ooev": "20241212-0156_gtfs_ooevv_2024", #upper austria
           "esg": "20241203-0058_gtfs_esg_2024", #linz
           "verbund": "20241217-0310_gtfs_verbundlinie_2024", #styria
           "kaernter": "20241214-0253_gtfs_kaerntnerlinien_2024", #carinthia
           "salzburg": "20241217-0359_gtfs_salzburgverkehr_2024", #salzburg
           "vvt": "20241217-0436_gtfs_vvt_2024", #tyrol
           "vmobil": "20241212-0624_gtfs_vmobil_2024", #vorarlberg
           "oebb": "GTFS_2024_obb"} #oebb maybe 20241217-0222_gtfs_evu_2024

state_name = "vmobil"

# select day for calculation in format YYYYMMDD in 2024
selected_day = 20240530

# stop categories
table = np.array([
    ["I", "I", "II", "III"],        # < 5 min
    ["I", "II", "III", "III"],      # 5 >= x <= 10
    ["II", "III", "IV", "IV"],      # 10 < x < 20
    ["III", "IV", "V", "V"],        # 20 >= x < 40
    ["IV", "V", "VI", "VI"],        # 40 >= x <= 60
    ["V", "VI", "VII", "VII"],      # 60 < x <= 120  
    ["", "VII", "VIII", "VIII"],    # 120 < x <= 210 
    ["", "", "", ""],               # > 210

])

transport_category = ["Fernverkehr REX", 
                      "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
                      "Straßenbahn, Metrobus, 0-Bus", 
                      "Bus"]


In [7]:
def lookup_category(interval, t_cat):
    if interval < 5:
        return table[0][t_cat]
    elif interval <= 10:
        return table[1][t_cat]
    elif interval < 20:
        return table[2][t_cat]
    elif interval < 40:
        return table[3][t_cat]
    elif interval <= 60:
        return table[4][t_cat]
    elif interval <= 120:
        return table[5][t_cat]
    elif interval <= 210:
        return table[6][t_cat]
    else:
        return table[7][t_cat]

In [8]:

path = f"{data_path}/{regions[state_name]}/"

stops_df = pd.read_csv(path + "stops.txt", 
                        quotechar='"',
                        sep=",")

print(stops_df.head())

           stop_id                    stop_name   stop_lat   stop_lon  \
0   at:47:1222:0:4       St. Anton a.A. Bahnhof  47.127468  10.266639   
1    at:47:1222:22                    Steig 2+3  47.127450  10.267232   
2  at:47:61099:0:1    St. Anton a. A. Kohlereck  47.122358  10.253820   
3  at:47:61099:0:2    St. Anton a. A. Kohlereck  47.122322  10.253703   
4  at:47:62209:0:1  St. Anton a. A. Stadle B197  47.122273  10.248646   

   zone_id  location_type parent_station level_id platform_code  
0   6455.0            NaN    Pat:47:1222  Level 0             1  
1      NaN            NaN    Pat:47:1222  Level 0           NaN  
2   6455.0            NaN   Pat:47:61099  Level 0             1  
3   6455.0            NaN   Pat:47:61099  Level 0             2  
4   6455.0            NaN   Pat:47:62209  Level 0             1  


In [9]:
stop_times_df = pd.read_csv(path + "stop_times.txt", sep=',', quotechar='"')

# remove from stop_times_df entries outside of the time window (6am-8pm)
stop_times_df = stop_times_df[stop_times_df['departure_time'].between('06:00:00', '20:00:00')]

print(stop_times_df.head())

                     trip_id arrival_time departure_time         stop_id  \
27  1.T0.12-820-E-j24-1.20.R     06:00:00       06:00:00  at:48:1232:0:2   
28  1.T0.12-820-E-j24-1.20.R     06:01:00       06:01:00  at:48:1233:0:2   
29  1.T0.12-820-E-j24-1.20.R     06:02:00       06:02:00  at:48:1228:0:2   
30  1.T0.12-820-E-j24-1.20.R     06:03:00       06:03:00  at:48:1229:0:2   
31  1.T0.12-820-E-j24-1.20.R     06:05:00       06:05:00  at:48:1235:0:2   

    stop_sequence  stop_headsign  pickup_type  drop_off_type  \
27             28            NaN            0              0   
28             29            NaN            0              0   
29             30            NaN            0              0   
30             31            NaN            0              0   
31             32            NaN            0              0   

    shape_dist_traveled  
27             19261.74  
28             20149.77  
29             20924.20  
30             21372.79  
31             22450.46  


In [10]:
trips_df = pd.read_csv(path + 'trips.txt', sep=',', quotechar='"')
print(trips_df.shape)

(20739, 8)


In [11]:
calendar_df = pd.read_csv(path + "calendar.txt", sep=",", quotechar='"')

#calendar_dates_df = pd.read_csv(data_path + state_name + "calendar_dates.txt", sep=",", quotechar='"')

print(calendar_df)

            service_id  monday  tuesday  wednesday  thursday  friday  \
0                   T0       1        1          1         1       1   
1                 T0#1       1        1          1         1       1   
2             T0+02o00       1        1          1         1       1   
3             T0+05310       0        0          0         0       1   
4             T0+05p00       1        1          1         1       1   
..                 ...     ...      ...        ...       ...     ...   
842   TA-90-11-j24+T21       1        1          1         1       1   
843   TA-90-11-j24+T22       0        0          0         0       0   
844   TA-90-11-j24+T24       0        0          0         0       0   
845   TA-90-11-j24+T35       1        1          1         1       0   
846  TA-90-11-j24+T35!       0        0          0         0       1   

     saturday  sunday  start_date  end_date  
0           0       0    20231210  20241214  
1           0       0    20240708  20240913

In [12]:
# remove all services entries in calendar_df that are not relevant for the specified day

calendar_filtered_df = calendar_df[(calendar_df['start_date'] <= selected_day) & (calendar_df['end_date'] >= selected_day)]
# TODO: check exceptions from calendar_dates_df
print(calendar_filtered_df.shape)

(813, 10)


In [13]:
# remove entries from trips_df that are not valid
trips_filtered_df = trips_df[trips_df['service_id'].isin(calendar_filtered_df['service_id'])]
print(trips_filtered_df.shape)

(19548, 8)


In [14]:
print(stop_times_df.shape)

# keep only valid trips from trips_filtered_df
stop_times_df = stop_times_df[stop_times_df['trip_id'].isin(trips_filtered_df['trip_id'])]

print(stop_times_df.shape)

(347880, 9)
(342300, 9)


In [15]:
df = stops_df.merge(stop_times_df, on='stop_id')
result = df.groupby("parent_station").size().reset_index(name="count")
result.rename(columns={"parent_station": "stop_id"}, inplace=True)
print(result.head())

        stop_id  count
0   Pat:47:1222    174
1  Pat:47:61099    174
2  Pat:47:62209     87
3  Pat:47:62504     87
4  Pat:47:64938     69


In [16]:
# merge result with stops parent stations
stops_final_df = stops_df.merge(result, on='stop_id', how='right')
print(stops_final_df.head())

        stop_id                      stop_name   stop_lat   stop_lon  zone_id  \
0   Pat:47:1222   St. Anton am Arlberg Bahnhof  47.127389  10.266782      NaN   
1  Pat:47:61099      St. Anton a. A. Kohlereck  47.122346  10.253766      NaN   
2  Pat:47:62209    St. Anton a. A. Stadle B197  47.122267  10.248583      NaN   
3  Pat:47:62504        St. Anton a. A. Brandli  47.124901  10.260108      NaN   
4  Pat:47:64938  St. Anton a. A. Terminal West  47.126527  10.263333      NaN   

   location_type parent_station level_id platform_code  count  
0            1.0            NaN      NaN           NaN    174  
1            1.0            NaN      NaN           NaN    174  
2            1.0            NaN      NaN           NaN     87  
3            1.0            NaN      NaN           NaN     87  
4            1.0            NaN      NaN           NaN     69  


In [17]:
# calculate the PTSQL

stops_final_df["interval"] = stops_final_df["count"].apply(lambda x: 840 / (x/2))
display(stops_final_df.sort_values(by="interval", ascending=True))

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code,count,interval
1423,Pat:48:585,Dornbirn Bahnhof,47.417415,9.738591,NaN,1.0,NaN,NaN,NaN,4897,0.343067
1300,Pat:48:452,Bregenz Bahnhof,47.502342,9.739957,NaN,1.0,NaN,NaN,NaN,2643,0.635641
1534,Pat:48:741,Dornbirn Sägerbrücke/Campus V,47.406084,9.742265,NaN,1.0,NaN,NaN,NaN,1501,1.119254
1329,Pat:48:483,Bregenz Montfortstraße,47.502506,9.744789,NaN,1.0,NaN,NaN,NaN,1420,1.183099
1366,Pat:48:510,Bregenz Wolfeggstraße,47.500096,9.739669,NaN,1.0,NaN,NaN,NaN,1397,1.202577
...,...,...,...,...,...,...,...,...,...,...,...
934,Pat:48:2188,Lustenau Altes Feuerwehrhaus,47.429504,9.664031,NaN,1.0,NaN,NaN,NaN,1,1680.000000
1802,Pde:09776:2280,Simmerberg Rieder,47.579212,9.935762,NaN,1.0,NaN,NaN,NaN,1,1680.000000
848,Pat:48:2059,Brederis Volksschule,47.279125,9.609827,NaN,1.0,NaN,NaN,NaN,1,1680.000000
603,Pat:48:1726,Götzis Im Hag,47.343273,9.651967,NaN,1.0,NaN,NaN,NaN,1,1680.000000


In [18]:
# Betrachtungszeitraum: 6–20 Uhr (= 840 Minuten)
# Stichtage: Werktag ohne Schule (Herbstferien): im Jahr 2021 der 28. 10.
# Normaler Werktag mit Schule: im Jahr 2021 der 22. 10.
# Intervallberechnung: Bildung der Summe der
# Abfahrten aller Verkehrsmittel über alle Ver-
# kehrsmittelkategorien, Multiplikation mit einem
# Richtungsfaktor von 0,5 und Berechnung des
# durchschnittlichen Intervalls über den gesamten
# Betrachtungszeitraum pro Richtung (840 Minuten
# dividiert durch die Zahl der Abfahrten pro Rich-
# tung). Der Richtungsfaktor wird auf allen Linien
# angewendet, Rundlinien ebenfalls.

# idea:
# 0. select specific day
# 1. take trip_id from entry in stop_times_df
# 2. look for that trip_id in trips_df
# 3. check if service_id from corresponding trip entry in calender_df is in valid time range,
#    also check exceptions in calender_date_df
# 4. if stop is valid, add to data for stop_id parent station


In [19]:
# code to determine transport categories
# transport_category = ["Fernverkehr REX", 
                    #   "S-Bahn / U-Bahn, Regionalbahn, Schnellbus, Lokalbahn", 
                    #   "Straßenbahn, Metrobus, 0-Bus", 
                    #   "Bus"]

# {0: "Tram", 1: "Metro", 2: "Fernverkehr REX", 3: "Bus", 11: "Electric Bus", }
route_type_translation = {0: 2, 1: 1, 2: 0, 3: 3, 11: 3, }

In [20]:
routes_df = pd.read_csv(path + 'routes.txt', sep=',', quotechar='"')
display(routes_df[["route_id", "route_type"]].head())

,route_id,route_type
0,at:vvv:101:,3
1,at:vvv:102:,3
2,at:vvv:103:,3
3,at:vvv:104:,3
4,at:vvv:105:,3


In [21]:
display(trips_filtered_df.head())
stop_type_df = trips_filtered_df[["route_id", "trip_id"]].merge(routes_df[["route_id", "route_type"]], on='route_id', how='left')
display(stop_type_df.head())

,route_id,service_id,trip_id,shape_id,trip_headsign,trip_short_name,direction_id,block_id
0,at:vvv:101:,T0+qcp00,1.T0.31-101-E-j24-1.1.H,31-101-E-j24-1.1.H,Bregenz Pfänderbahn,NaN,0,NaN
1,at:vvv:101:,T2,1.T2.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
2,at:vvv:101:,T3,1.T3.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
3,at:vvv:101:,T0,10.T0.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN
4,at:vvv:101:,T2,10.T2.31-101-E-j24-1.2.H,31-101-E-j24-1.2.H,Bregenz Bahnhof,NaN,0,NaN


,route_id,trip_id,route_type
0,at:vvv:101:,1.T0.31-101-E-j24-1.1.H,3
1,at:vvv:101:,1.T2.31-101-E-j24-1.2.H,3
2,at:vvv:101:,1.T3.31-101-E-j24-1.2.H,3
3,at:vvv:101:,10.T0.31-101-E-j24-1.2.H,3
4,at:vvv:101:,10.T2.31-101-E-j24-1.2.H,3


In [22]:
display(stop_times_df.head())
stop_type_df = stop_type_df[["trip_id", "route_type"]].merge(stop_times_df[['stop_id', 'trip_id']], on='trip_id', how='left')
display(stop_type_df.head())

,trip_id,arrival_time,departure_time,stop_id,stop_sequence,stop_headsign,pickup_type,drop_off_type,shape_dist_traveled
27,1.T0.12-820-E-j24-1.20.R,06:00:00,06:00:00,at:48:1232:0:2,28,NaN,0,0,19261.74
28,1.T0.12-820-E-j24-1.20.R,06:01:00,06:01:00,at:48:1233:0:2,29,NaN,0,0,20149.77
29,1.T0.12-820-E-j24-1.20.R,06:02:00,06:02:00,at:48:1228:0:2,30,NaN,0,0,20924.20
30,1.T0.12-820-E-j24-1.20.R,06:03:00,06:03:00,at:48:1229:0:2,31,NaN,0,0,21372.79
31,1.T0.12-820-E-j24-1.20.R,06:05:00,06:05:00,at:48:1235:0:2,32,NaN,0,0,22450.46


,trip_id,route_type,stop_id
0,1.T0.31-101-E-j24-1.1.H,3,at:48:452:0:5
1,1.T0.31-101-E-j24-1.1.H,3,at:48:469:0:2
2,1.T0.31-101-E-j24-1.1.H,3,at:48:466:0:2
3,1.T0.31-101-E-j24-1.1.H,3,at:48:471:0:2
4,1.T0.31-101-E-j24-1.1.H,3,at:48:501:0:1


In [23]:
display(stops_df.head())
stop_type_df = stop_type_df[["stop_id", "route_type"]].merge(stops_df[["stop_id", "parent_station"]], on="stop_id", how="left")
display(stop_type_df.head())
display(stop_type_df.shape)

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code
0,at:47:1222:0:4,St. Anton a.A. Bahnhof,47.127468,10.266639,6455.0,NaN,Pat:47:1222,Level 0,1
1,at:47:1222:22,Steig 2+3,47.127450,10.267232,NaN,NaN,Pat:47:1222,Level 0,NaN
2,at:47:61099:0:1,St. Anton a. A. Kohlereck,47.122358,10.253820,6455.0,NaN,Pat:47:61099,Level 0,1
3,at:47:61099:0:2,St. Anton a. A. Kohlereck,47.122322,10.253703,6455.0,NaN,Pat:47:61099,Level 0,2
4,at:47:62209:0:1,St. Anton a. A. Stadle B197,47.122273,10.248646,6455.0,NaN,Pat:47:62209,Level 0,1


,stop_id,route_type,parent_station
0,at:48:452:0:5,3,Pat:48:452
1,at:48:469:0:2,3,Pat:48:469
2,at:48:466:0:2,3,Pat:48:466
3,at:48:471:0:2,3,Pat:48:471
4,at:48:501:0:1,3,Pat:48:501


(343881, 3)

In [24]:
station_type_df = stop_type_df.copy()
station_type_df["rank"] = station_type_df["route_type"].apply(lambda x: route_type_translation[x]).min()
station_type_df = station_type_df.groupby("parent_station")["rank"].min().reset_index(name="rank")
display(station_type_df.head())
display(station_type_df.shape)

,parent_station,rank
0,Pat:47:1222,3
1,Pat:47:61099,3
2,Pat:47:62209,3
3,Pat:47:62504,3
4,Pat:47:64938,3


(2008, 2)

In [25]:
station_type_df.rename(columns={'parent_station':'stop_id'}, inplace=True)
stops_final_df = stops_final_df.merge(station_type_df, on='stop_id', how='left')
display(stops_final_df.head())

,stop_id,stop_name,stop_lat,stop_lon,zone_id,location_type,parent_station,level_id,platform_code,count,interval,rank
0,Pat:47:1222,St. Anton am Arlberg Bahnhof,47.127389,10.266782,NaN,1.0,NaN,NaN,NaN,174,9.655172,3
1,Pat:47:61099,St. Anton a. A. Kohlereck,47.122346,10.253766,NaN,1.0,NaN,NaN,NaN,174,9.655172,3
2,Pat:47:62209,St. Anton a. A. Stadle B197,47.122267,10.248583,NaN,1.0,NaN,NaN,NaN,87,19.310345,3
3,Pat:47:62504,St. Anton a. A. Brandli,47.124901,10.260108,NaN,1.0,NaN,NaN,NaN,87,19.310345,3
4,Pat:47:64938,St. Anton a. A. Terminal West,47.126527,10.263333,NaN,1.0,NaN,NaN,NaN,69,24.347826,3


In [26]:
print(lookup_category(9.655172, 3))

III
